In [1]:
import duckdb
import os

# Change to project root so relative paths work
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

# Create a persistent database file in data/processed/
# (In-memory would lose data when the kernel restarts)
db_path = "data/processed/quickbite.duckdb"
conn = duckdb.connect(db_path)
print(f"Connected to DuckDB at: {db_path}")

Working directory: c:\Users\piriy\Downloads\Quickbite-delivery-Analysis
Connected to DuckDB at: data/processed/quickbite.duckdb


In [2]:
# Read the SQL file and execute it
with open('sql/01_create_schema.sql', 'r') as f:
    schema_sql = f.read()

conn.execute(schema_sql)
print("Schema created successfully")

# List the tables
tables = conn.execute("SHOW TABLES").fetchall()
print("Tables in database:")
for t in tables:
    print(f"  - {t[0]}")

Schema created successfully
Tables in database:
  - dim_conditions
  - dim_courier
  - dim_date
  - dim_location
  - dim_restaurant
  - fact_orders


In [3]:
with open('sql/02_load_data.sql', 'r') as f:
    load_sql = f.read()

conn.execute(load_sql)
print("Data loaded successfully")

Data loaded successfully


In [4]:
with open('sql/03_verify_counts.sql', 'r') as f:
    verify_sql = f.read()

result = conn.execute(verify_sql).fetchdf()
print(result)

       table_name  row_count
0     fact_orders      39141
1     dim_courier       1170
2  dim_restaurant        388
3        dim_date         44
4    dim_location          3
5  dim_conditions         24


In [5]:
# Your first real analytical query — no pandas, just SQL
sample_query = """
SELECT 
    c.weather,
    c.traffic,
    COUNT(*) AS order_count,
    ROUND(AVG(f.delivery_minutes), 1) AS avg_delivery_min,
    ROUND(100.0 * SUM(f.is_on_time) / COUNT(*), 1) AS on_time_pct
FROM fact_orders f
JOIN dim_conditions c ON f.condition_id = c.condition_id
GROUP BY c.weather, c.traffic
ORDER BY avg_delivery_min DESC
LIMIT 10;
"""

result = conn.execute(sample_query).fetchdf()
print(result)

      weather traffic  order_count  avg_delivery_min  on_time_pct
0         Fog     Jam         2153              36.7         23.6
1      Cloudy     Jam         2063              36.6         24.9
2       Windy     Jam         2073              30.2         54.1
3      Stormy     Jam         2039              30.0         56.0
4  Sandstorms     Jam         2089              30.0         55.0
5      Cloudy    High          651              29.0         62.2
6      Cloudy  Medium         1603              28.6         50.8
7         Fog    High          673              28.5         64.6
8         Fog  Medium         1619              28.4         52.7
9  Sandstorms    High          605              27.9         62.1


In [6]:
conn.close()
print("Database connection closed")

Database connection closed
